In [1]:
import bldw
import numpy as np
import blimpy as bl
import pandas as pd
import matplotlib.pyplot as plt
import os
import glob
import setigen as stg
from astropy import units as u
%matplotlib inline

plt.rcParams["font.family"] = "serif"
plt.rcParams["mathtext.fontset"] = "dejavuserif"

numexpr.utils   INFO     Note: NumExpr detected 40 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 8.
numexpr.utils   INFO     NumExpr defaulting to 8 threads.


In [2]:
dirstring = '/datax/scratch/benjb/bl_nearby_stars/blpc1_spliced/*.dat'

ivec = []

count_empty = 0
for i, file in enumerate(glob.iglob(dirstring)):
    if i%100000 == 0:
        print(i)
    size_bytes = os.path.getsize(file)
    if size_bytes==0:
        print('found empty')
        count_empty += 1
    index = os.path.basename(file).split('_')[0]
    ivec.append(index)

# filename starts with {index}_{h5idx}_{k}

# k is the coarse channel number

0
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
found empty
100000
found empty
found empty
found empty
found empty
found empty
found e

KeyboardInterrupt: 

In [3]:
print(count_empty/len(ivec))
print(count_empty, len(ivec))

0.0011179694743265942
7903 7069066


In [ ]:
# dirstring = '/datax/scratch/benjb/bl_nearby_stars/bliss_dats_pooled_090825/*.dat'

# ivec2 = []

# for i, file in enumerate(glob.iglob(dirstring)):
#     if i%100000 == 0:
#         print(i)
#     index = os.path.basename(file).split('_')[0]
#     ivec2.append(index)

# # filename starts with {index}_{h5idx}_{k}

# # k is the coarse channel number

0
100000


In [4]:
u = np.sort(np.unique(ivec).astype(int))
# u2 = np.sort(np.unique(ivec2).astype(int))

In [ ]:
# np.savez('/datax/scratch/benjb/bl_nearby_stars/blpc2_spliced_files_idx_092925.npz', u)

In [5]:
print(len(u))

1559


In [11]:
print(list(u))
#print(list(u2))

[0, 1, 2, 63, 64, 65, 95, 96, 97, 121, 122, 123, 147, 148, 149, 156, 157, 171, 172, 9178, 9179, 9180, 9197, 9198, 9199, 9200, 9201, 9202, 9203, 9204, 9205, 9206, 9207, 9208, 9209, 9210, 9241, 9242, 19474, 19475, 19499, 19571, 19578, 19579, 19580, 19581, 19582, 19583, 19584, 19585, 25920, 25944, 25945, 25946, 25947, 26000, 26001, 26002, 26003, 26004, 26005, 26006, 26044, 26045, 26046, 26047, 26048, 26049, 26050, 26051, 26052, 26053, 26085, 26086, 26087, 26088, 26089, 26090, 26091, 26092, 26093, 26094, 26118, 26119, 26120, 26121, 26122, 26123, 26153, 26154, 26155, 26205, 26206, 26207, 26239, 26240, 26241, 26242, 26267, 26268, 26269, 26293, 26294, 26295, 26296, 26320, 26321, 26322, 26346, 26347, 26383, 26384, 26385, 26386, 26387, 26411, 26412, 26413, 26414, 26415, 26439, 26440, 26441, 26442, 26443, 26496, 26497, 26498, 26522, 26523, 26524, 26525, 26526, 26527, 26528, 26552, 26553, 26554, 26642, 26643, 26667, 26668, 26669, 26670, 26671, 26709, 26710, 26711, 26764, 26765, 26766, 26798, 2679

In [6]:
def combine_hits(list_of_dats):
    cadvec = []
    scanvec = []
    ccvec = []
    for dat in list_of_dats:
        spl = os.path.basename(dat).split('_')
        cadvec.append(spl[0])
        scanvec.append(spl[1])
        ccvec.append(spl[2])
    ccvec = np.array(ccvec).astype('int')
    cadi = np.unique(cadvec)
    scani = np.unique(scanvec)
    if len(cadi) > 1:
        print(f'Too many cadence indices: {cadi}')
    if len(scani) > 1:
        print(f'Too many scan indices: {scani}')
    if np.isin(np.arange(len(ccvec)), ccvec).all():
        print(f'{cadi}: All coarse channels accounted for!')
    else:
        print(f'{cadi}: Missing coarse channels:', ccvec[~np.isin(np.arange(len(ccvec)), ccvec)])
    ### write header
    size_list = np.array([os.path.getsize(file) for file in list_of_dats[:10]])
    j = np.where(size_list > 0)[0][0]
    with open(list_of_dats[j]) as f:
        print(list_of_dats[j])
        header = [next(f) for _ in range(9)]
    with open(f'/datax/scratch/benjb/bl_nearby_stars/respliced_dats_from_092925/{os.path.basename(list_of_dats[0])}', 'w') as f:
        for hline in header:
            f.write(hline)
    ### write hits
    for j in range(len(list_of_dats)):
        with open(list_of_dats[j]) as f:
            all_lines = f.readlines()
            if len(all_lines) > 0:
                lines = all_lines[9:]
            else:
                lines = []
        with open(f'/datax/scratch/benjb/bl_nearby_stars/respliced_dats_from_092925/{os.path.basename(list_of_dats[0])}', 'a') as f:
            for line in lines:
                f.write(line)
    

In [14]:
for i, ui in enumerate(u):
    if ui < 39113:
        continue
    cadfiles = glob.glob(f'/datax/scratch/benjb/bl_nearby_stars/bliss_dats_spliced_092925/{ui}_*.dat')
    print(np.sort(np.array(cadfiles)))
    ulen = len(str(ui)) # number of digits in index
    #print(len(files))
    for j in range(6):
        print(ui, j)
        # go through all files and put the ones from, e.g., scan 1 into a separate list
        scanfiles = []
        for cadfile in cadfiles:
            if int(os.path.basename(cadfile)[ulen+1])-1 == j:
                scanfiles.append(cadfile)
        scanfiles = np.sort(np.array(scanfiles))
        # write new function to put all hits into new dat file
        print(scanfiles)
        combine_hits(scanfiles)
        #print(scanfiles[-1])
        #break
    #break

['/datax/scratch/benjb/bl_nearby_stars/bliss_dats_spliced_092925/39113_1_0_spliced_blc00010203040506o7o0111213141516o7o0212223242526o7o031323334353637_guppi_58165_84689_SO0253_0010.gpuspec.0000_nosig_nosk_SNR_20_L1_30.dat'
 '/datax/scratch/benjb/bl_nearby_stars/bliss_dats_spliced_092925/39113_1_10_spliced_blc00010203040506o7o0111213141516o7o0212223242526o7o031323334353637_guppi_58165_84689_SO0253_0010.gpuspec.0000_nosig_nosk_SNR_20_L1_30.dat'
 '/datax/scratch/benjb/bl_nearby_stars/bliss_dats_spliced_092925/39113_1_11_spliced_blc00010203040506o7o0111213141516o7o0212223242526o7o031323334353637_guppi_58165_84689_SO0253_0010.gpuspec.0000_nosig_nosk_SNR_20_L1_30.dat'
 '/datax/scratch/benjb/bl_nearby_stars/bliss_dats_spliced_092925/39113_1_12_spliced_blc00010203040506o7o0111213141516o7o0212223242526o7o031323334353637_guppi_58165_84689_SO0253_0010.gpuspec.0000_nosig_nosk_SNR_20_L1_30.dat'
 '/datax/scratch/benjb/bl_nearby_stars/bliss_dats_spliced_092925/39113_1_13_spliced_blc00010203040506o7o0

IndexError: index 0 is out of bounds for axis 0 with size 0

In [ ]:
# check unspliced dats -- needs to be done separately for each blpc{0,1,2,3}
# check spliced dats -- currently doing for blpc2, needs to be done for blpc{0,1,3}
# merge all spliced dats -- currently doing for blpc2, needs to be done for blpc{0,1,3}
# pool dats together on a single blpc machine (likely blpc2)
# update CSV
# read off hits and make plots!
# start FindEvent

# figure out ad hoc plotting code for large turboSETI event counts (plot_large_dat_files.ipynb)